# Step 1: Data Collection and Understanding

In [14]:
import pandas as pd
import yfinance as yf

# Download Stock Data

ticker = "RELIANCE.NS"

data = yf.download(
    ticker,
    start="2020-01-01",
    interval="1d",
    auto_adjust=False
)

# Flatten MultiIndex Columns

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# Save Raw Dataset

data.to_csv("nse_ohlcv_raw.csv")

# Load Dataset

data = pd.read_csv("nse_ohlcv_raw.csv")

# Data Understanding

print("\nFirst 5 Rows")
print(data.head())

print("\nLast 5 Rows")
print(data.tail())

print("\nDataset Information")
data.info()

print("\nStatistical Summary")
print(data.describe())

print("\nColumn Names")
print(data.columns.tolist())

print("\nDataset Shape")
print(data.shape)

print("\nData Types")
print(data.dtypes)

[*********************100%***********************]  1 of 1 completed


First 5 Rows
         Date   Adj Close       Close        High         Low        Open  \
0  2020-01-01  672.216125  690.138306  698.138733  688.263977  693.978516   
1  2020-01-02  683.660278  701.887512  704.470520  691.235535  691.235535   
2  2020-01-03  684.484131  702.733276  704.790527  696.264343  700.835999   
3  2020-01-06  668.609314  686.435303  698.504456  684.835205  694.892883   
4  2020-01-07  678.895630  696.995850  701.521790  691.921265  694.435669   

     Volume  
0  14004468  
1  17710316  
2  20984698  
3  24519177  
4  16683622  

Last 5 Rows
            Date    Adj Close        Close         High          Low  \
1658  2026-09-07  1309.500000  1309.500000  1324.199951  1301.900024   
1659  2026-09-08  1294.900024  1294.900024  1306.800049  1288.000000   
1660  2026-09-09  1279.000000  1279.000000  1294.699951  1277.199951   
1661  2026-09-10  1274.000000  1274.000000  1285.300049  1266.300049   
1662  2026-09-11  1257.500000  1257.500000  1267.400024  1253.0000

# Step 2: Data Preprocessing

In [15]:

# Check Missing Values

print("\nMissing Values")
print(data.isnull().sum())

# Check Duplicate Rows

print("\nDuplicate Rows")
print(data.duplicated().sum())

# Convert Date Column to Datetime

data["Date"] = pd.to_datetime(data["Date"])

# Sort Data by Date

data = data.sort_values("Date")

# Remove Missing Values

data = data.dropna()

# Remove Duplicate Rows

data = data.drop_duplicates()

# Reset Index

data = data.reset_index(drop=True)

# Verify Dataset After Cleaning

print("\nAfter Cleaning")

print("\nMissing Values")
print(data.isnull().sum())

print("\nDuplicate Rows")
print(data.duplicated().sum())

print("\nDataset Shape")
print(data.shape)

# Save Clean Dataset

data.to_csv(
    "nse_ohlcv_clean.csv",
    index=False
)




Missing Values
Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

Duplicate Rows
0

After Cleaning

Missing Values
Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

Duplicate Rows
0

Dataset Shape
(1663, 7)


# Step 3: Feature Engineering


In [16]:

#Candle Body
data["Body"] = abs(data["Close"] - data["Open"])

#Total Candle Range
data["Range"] = data["High"] - data["Low"]

#Upper Wick
data["Upper_Wick"] = (
    data["High"]
    - data[["Open", "Close"]].max(axis=1)
)

#Lower Wick
data["Lower_Wick"] = (
    data[["Open", "Close"]].min(axis=1)
    - data["Low"]
)

#Ratios
data["Body_Ratio"] = data["Body"] / data["Range"]

data["Upper_Wick_Ratio"] = (
    data["Upper_Wick"] / data["Range"]
)

data["Lower_Wick_Ratio"] = (
    data["Lower_Wick"] / data["Range"]
)

#Replace infinite values if any
data.replace([float("inf"), -float("inf")], 0, inplace=True)

#Fill NaN values
data.fillna(0, inplace=True)

#Display Result
print("Feature Engineering Completed\n")

print(data[
    [
        "Open",
        "High",
        "Low",
        "Close",
        "Body",
        "Range",
        "Upper_Wick",
        "Lower_Wick"
    ]
].head())

#Save Dataset
data.to_csv(
    "candlestick_features.csv",
    index=False
)


Feature Engineering Completed

         Open        High         Low       Close       Body      Range  \
0  693.978516  698.138733  688.263977  690.138306   3.840210   9.874756   
1  691.235535  704.470520  691.235535  701.887512  10.651978  13.234985   
2  700.835999  704.790527  696.264343  702.733276   1.897278   8.526184   
3  694.892883  698.504456  684.835205  686.435303   8.457581  13.669250   
4  694.435669  701.521790  691.921265  696.995850   2.560181   9.600525   

   Upper_Wick  Lower_Wick  
0    4.160217    1.874329  
1    2.583008    0.000000  
2    2.057251    4.571655  
3    3.611572    1.600098  
4    4.525940    2.514404  


# STEP 4 : MARKET STRUCTURE AND TREND ANALYSIS

In [20]:




# =====================================================
# SWING HIGH AND SWING LOW DETECTION
# =====================================================

left = 2
right = 2

data["Swing_High"] = False
data["Swing_Low"] = False

for i in range(left, len(data) - right):

    current_high = data["High"].iloc[i]
    current_low = data["Low"].iloc[i]

    left_highs = data["High"].iloc[i-left:i]
    right_highs = data["High"].iloc[i+1:i+right+1]

    left_lows = data["Low"].iloc[i-left:i]
    right_lows = data["Low"].iloc[i+1:i+right+1]

    # Swing High

    if (
        current_high > left_highs.max()
        and
        current_high > right_highs.max()
    ):
        data.loc[i, "Swing_High"] = True

    # Swing Low

    if (
        current_low < left_lows.min()
        and
        current_low < right_lows.min()
    ):
        data.loc[i, "Swing_Low"] = True



# =====================================================
# HH HL LH LL IDENTIFICATION
# =====================================================

data["HH"] = 0
data["HL"] = 0
data["LH"] = 0
data["LL"] = 0

previous_swing_high = None
previous_swing_low = None

for i in range(len(data)):

    # Swing High Logic

    if data["Swing_High"].iloc[i]:

        current_high = data["High"].iloc[i]

        if previous_swing_high is not None:

            if current_high > previous_swing_high:
                data.loc[i, "HH"] = 1

            elif current_high < previous_swing_high:
                data.loc[i, "LH"] = 1

        previous_swing_high = current_high

    # Swing Low Logic

    if data["Swing_Low"].iloc[i]:

        current_low = data["Low"].iloc[i]

        if previous_swing_low is not None:

            if current_low > previous_swing_low:
                data.loc[i, "HL"] = 1

            elif current_low < previous_swing_low:
                data.loc[i, "LL"] = 1

        previous_swing_low = current_low



# =====================================================
# COUNT STRUCTURE EVENTS
# LAST 10 CANDLES
# =====================================================

window_size = 10

data["HH_Count"] = (
    data["HH"]
    .rolling(window_size)
    .sum()
)

data["HL_Count"] = (
    data["HL"]
    .rolling(window_size)
    .sum()
)

data["LH_Count"] = (
    data["LH"]
    .rolling(window_size)
    .sum()
)

data["LL_Count"] = (
    data["LL"]
    .rolling(window_size)
    .sum()
)

# =====================================================
# TREND IDENTIFICATION
# =====================================================

def identify_trend(row):

    if pd.isna(row["HH_Count"]):
        return "Not Available"

    # Uptrend

    elif (
        row["HH_Count"] > row["LH_Count"]
        and
        row["HL_Count"] > row["LL_Count"]
    ):
        return "Uptrend"

    # Downtrend

    elif (
        row["LH_Count"] > row["HH_Count"]
        and
        row["LL_Count"] > row["HL_Count"]
    ):
        return "Downtrend"

    # Sideways

    else:
        return "Sideways"

data["Trend"] = data.apply(
    identify_trend,
    axis=1
)



# =====================================================
# MARKET CONTROL
# =====================================================

def market_control(row):

    if row["Trend"] == "Uptrend":
        return "Buyers"

    elif row["Trend"] == "Downtrend":
        return "Sellers"

    else:
        return "Neutral"

data["Market_Control"] = data.apply(
    market_control,
    axis=1
)



# =====================================================
# HIGHER TIMEFRAME BIAS
# =====================================================

def htf_bias(row):

    if row["Trend"] == "Uptrend":
        return "Bullish"

    elif row["Trend"] == "Downtrend":
        return "Bearish"

    else:
        return "Neutral"

data["HTF_Bias"] = data.apply(
    htf_bias,
    axis=1
)



# =====================================================
# DISPLAY LATEST MARKET STRUCTURE
# =====================================================

latest = data.iloc[-1]

print("\nLatest Market Structure\n")

print("Date :", latest["Date"])

print("Trend :", latest["Trend"])

print("Market Control :", latest["Market_Control"])

print("HTF Bias :", latest["HTF_Bias"])

print("HH Count :", latest["HH_Count"])

print("HL Count :", latest["HL_Count"])

print("LH Count :", latest["LH_Count"])

print("LL Count :", latest["LL_Count"])

# =====================================================
# TREND DISTRIBUTION
# =====================================================

print("\nTrend Distribution\n")

print(
    data["Trend"].value_counts()
)

# =====================================================
# SAVE DATASET
# =====================================================

data.to_csv(
    "market_structure_analysis.csv",
    index=False
)




Latest Market Structure

Date : 2026-09-11 00:00:00
Trend : Sideways
Market Control : Neutral
HTF Bias : Neutral
HH Count : 1.0
HL Count : 0.0
LH Count : 0.0
LL Count : 1.0

Trend Distribution

Trend
Sideways         992
Uptrend          388
Downtrend        274
Not Available      9
Name: count, dtype: int64
